# 05 — Training progress and independent GNU Go evaluation

This notebook reads a completed AHR v2 checkpoint, shows training progress, and measures development games against GNU Go level 1. Run All performs short correctness fixtures; set `RUN_EVALUATION = True` to run the arena. It never trains a model or opens final/lockbox games.

**Inputs:** notebooks 00–04 through `ahr-export` definitions, a training checkpoint, and GNU Go 3.8. **Outputs:** `reports/ahr_v2/evaluations/<eval_id>/`, plus resumable arena checkpoints in `checkpoints/<eval_id>/`. **Expected runtime:** under two minutes for the default validation; an eight-game arena depends on game length and search budget and may take tens of minutes.

Both colors use 9×9, komi 7.5, area scoring, positional superko and no suicide. GNU Go is launched at level 1 with `--capture-all-dead` and `--never-resign`; all captures must be played out. Final results use our independently validated rules, with no automatic dead-stone removal. Timeout, illegal opponent moves, resignations and ply cutoffs are **UNKNOWN**. Search values are estimates, not proofs of full-game strength.

A model snapshot, source identities, opponent executable hash and all game/search state are pinned in the arena checkpoint. Only complete games enter its win/loss/draw denominator. UNKNOWN attempts are shown separately and in a conservative all-attempts win bound. Small development reviews do not establish a final playing-strength claim.

## Setup and controls

In [ ]:
from pathlib import Path
import nbformat
PROJECT = Path.cwd().resolve()
while not (PROJECT / 'AGENTS.md').is_file() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
for cell in nbformat.read(PROJECT / 'experiments/ahr_v2/00_contracts_and_recovery.ipynb', 4).cells:
    if cell.cell_type == 'code' and 'ahr-export' in cell.metadata.get('tags', []):
        exec(compile(cell.source, '00:ahr-export', 'exec'), globals())
for name in ('01_rules_and_replay.ipynb', '02_scout_worker_network.ipynb', '03_batched_puct_search.ipynb'):
    go2_import_exports(PROJECT / GO2_ROOT / name, globals())
training_notebook = PROJECT / GO2_ROOT / '04_selfplay_and_training.ipynb'
if training_notebook.is_file():
    go2_import_exports(training_notebook, globals())
RUN_EVALUATION = False
RUN_ID = None                       # Resolve the active AHR v2 run when None.
EVAL_ID = None                      # Keep the same value to resume an existing review.
DEVICE = 'cpu'                     # Leave the training GPU available.


## Arena configuration and GTP transport

In [ ]:
import collections, dataclasses, math, selectors, shutil, subprocess

@dataclasses.dataclass(frozen=True)
class Go2EvalConfig:
    games: int = 8
    board_size: int = 9
    komi: float = 7.5
    simulations: int = 32
    batch_size: int = 8
    seed: int = 2026100605
    max_plies: int = 500
    game_time_seconds: float | None = 900.0
    move_time_seconds: float | None = None
    gtp_timeout_seconds: float = 30.0
    purpose: str = 'development'
    def __post_init__(self):
        if self.games < 2 or self.games % 2:
            raise ValueError('Use an even number of at least two games for balanced colors.')
        if not 2 <= self.board_size <= 19 or self.simulations < 1 or self.batch_size < 1 or self.max_plies < 2:
            raise ValueError('Invalid arena board/search/ply budget.')
        if not math.isfinite(self.komi) or self.gtp_timeout_seconds <= 0:
            raise ValueError('Invalid komi or GTP timeout.')
        if self.game_time_seconds is not None and self.game_time_seconds <= 0:
            raise ValueError('Positive game timeout required.')
        if self.move_time_seconds is not None and self.move_time_seconds <= 0:
            raise ValueError('Positive move timeout required.')
        if self.purpose not in ('development', 'validation_fixture'):
            raise ValueError('This notebook only opens development or disposable validation games.')

def go2_gtp_vertex(action, size):
    if action == size * size: return 'pass'
    if type(action) is not int or not 0 <= action < size * size: raise ValueError('Invalid action.')
    row, col = divmod(action, size)
    return 'ABCDEFGHJKLMNOPQRST'[col] + str(size - row)

def go2_gtp_action(vertex, size):
    value = vertex.strip().upper()
    if value == 'PASS': return size * size
    if value == 'RESIGN': raise Go2OpponentError('GNU Go resignation is UNKNOWN, not a completed exact game.')
    if len(value) < 2 or value[0] not in 'ABCDEFGHJKLMNOPQRST'[:size]: raise ValueError('Bad GTP vertex.')
    try: row = int(value[1:])
    except ValueError as exc: raise ValueError('Bad GTP row.') from exc
    if not 1 <= row <= size: raise ValueError('GTP vertex outside board.')
    return (size - row) * size + 'ABCDEFGHJKLMNOPQRST'.index(value[0])

class Go2OpponentError(RuntimeError): pass

class Go2GTP:
    """Bounded binary transport. A restarted engine reproduces every past genmove."""
    def __init__(self, config, seed, executable=None):
        self.config = config
        self.path = Path(executable or shutil.which('gnugo') or '/usr/games/gnugo').resolve()
        if not self.path.is_file(): raise FileNotFoundError('GNU Go dependency missing; install the gnugo package.')
        self.flags = ['--mode', 'gtp', '--level', '1', '--seed', str(seed), '--chinese-rules',
                      '--positional-superko', '--forbid-suicide', '--never-resign', '--capture-all-dead']
        self.process = subprocess.Popen([str(self.path), *self.flags], stdin=subprocess.PIPE,
                                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=0)
        self.selector = selectors.DefaultSelector(); self.selector.register(self.process.stdout, selectors.EVENT_READ)
        self.buffer = bytearray(); self.sequence = 0
        try:
            self.version = self.command('version')
            self.command('boardsize ' + str(config.board_size)); self.command('clear_board')
            self.command('komi ' + str(config.komi))
        except BaseException:
            self.close(); raise
    def command(self, text):
        self.sequence += 1; number = self.sequence
        if self.process.poll() is not None: raise Go2OpponentError('GNU Go exited before a response.')
        self.process.stdin.write((str(number) + ' ' + text + '\n').encode()); self.process.stdin.flush()
        deadline = time.monotonic() + self.config.gtp_timeout_seconds
        while b'\n\n' not in self.buffer:
            remaining = deadline - time.monotonic()
            if remaining <= 0: raise TimeoutError('GNU Go response timeout: ' + text.split()[0])
            if not self.selector.select(remaining): raise TimeoutError('GNU Go response timeout.')
            chunk = os.read(self.process.stdout.fileno(), 65536)
            if not chunk: raise Go2OpponentError('GNU Go closed its output.')
            self.buffer.extend(chunk.replace(b'\r\n', b'\n'))
        raw, _, tail = self.buffer.partition(b'\n\n'); self.buffer = bytearray(tail)
        response = raw.decode('utf-8').strip()
        prefix = '=' + str(number)
        if not response.startswith(prefix) or (len(response) > len(prefix) and not response[len(prefix)].isspace()):
            raise Go2OpponentError('GTP rejected/mismatched command: ' + response[:200])
        return response[len(prefix):].strip()
    def play(self, color, action):
        self.command('play ' + ('black' if color == 1 else 'white') + ' ' + go2_gtp_vertex(action, self.config.board_size))
    def genmove(self, color):
        response = self.command('genmove ' + ('black' if color == 1 else 'white'))
        try: return go2_gtp_action(response, self.config.board_size)
        except (ValueError, RuntimeError) as exc: raise Go2OpponentError(str(exc)) from exc
    def replay(self, trace):
        for item in trace:
            if item['kind'] == 'play': self.play(item['color'], item['action'])
            elif item['kind'] == 'genmove':
                if self.genmove(item['color']) != item['action']:
                    raise Go2OpponentError('GNU Go regenerated a different move; exact resume refused.')
            else: raise RuntimeError('Invalid opponent replay command.')
    def close(self):
        if getattr(self, 'process', None) is not None:
            if self.process.poll() is None:
                self.process.terminate()
                try: self.process.wait(timeout=2)
                except subprocess.TimeoutExpired: self.process.kill(); self.process.wait(timeout=2)
            for stream in (self.process.stdin, self.process.stdout):
                if stream is not None: stream.close()
        if getattr(self, 'selector', None) is not None: self.selector.close()
    def __enter__(self): return self
    def __exit__(self, *_): self.close()

class _Go2EvalPause(Exception): pass

def go2_eval_sources(project):
    project = Path(project)
    sources = go2_source_identity(project)
    name = GO2_ROOT + '/05_evaluation_and_progress.ipynb'
    sources[name] = go2_code_sha(project / name)
    return sources

def go2_opponent_pin(config):
    executable = Path(shutil.which('gnugo') or '/usr/games/gnugo').resolve()
    if not executable.is_file(): raise FileNotFoundError('GNU Go not installed.')
    return {'name': 'GNU Go', 'level': 1, 'executable': str(executable), 'sha256': go2_file_sha(executable),
            'rules': 'chinese_no_suicide_positional_superko_capture_all_dead_never_resign',
            'version': subprocess.run([str(executable), '--version'], capture_output=True, text=True, check=True).stdout.splitlines()[0]}


## Denominators and read-only progress

In [ ]:
def go2_wilson(successes, total, z=1.959963984540054):
    if type(successes) is not int or type(total) is not int or not 0 <= successes <= total:
        raise ValueError('Integer successes must lie between zero and total.')
    if total == 0: return [None, None]
    p = successes / total; denominator = 1 + z*z/total
    midpoint = (p + z*z/(2*total)) / denominator
    spread = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / denominator
    return [max(0.0, midpoint-spread), min(1.0, midpoint+spread)]

def go2_eval_summary(results):
    counts = collections.Counter(item['outcome'] for item in results)
    if set(counts) - {'WIN', 'LOSS', 'DRAW', 'UNKNOWN'}: raise ValueError('Unexpected arena result.')
    attempts = len(results); complete = counts['WIN'] + counts['LOSS'] + counts['DRAW']
    wins = counts['WIN']; unknown = counts['UNKNOWN']
    rate = wins / complete if complete else None
    interval = go2_wilson(wins, complete)
    by_color = {}
    for color, label in ((1, 'black'), (-1, 'white')):
        items = [item for item in results if item['ahr_color'] == color]
        color_counts = collections.Counter(item['outcome'] for item in items)
        by_color[label] = {key.lower(): color_counts[key] for key in ('WIN', 'LOSS', 'DRAW', 'UNKNOWN')}
        by_color[label]['attempts'] = len(items)
    return {'attempts': attempts, 'complete_games': complete, 'wins': wins, 'losses': counts['LOSS'],
            'draws': counts['DRAW'], 'unknown': unknown, 'win_rate_complete_games': rate,
            'win_rate_wilson95_complete_games': interval,
            'win_rate_all_attempts_lower_bound': wins/attempts if attempts else None,
            'win_rate_all_attempts_upper_bound': (wins+unknown)/attempts if attempts else None,
            'observed_half_target': bool(attempts and not unknown and 2*wins >= attempts),
            'wilson_lower_at_least_half': bool(interval[0] is not None and not unknown and interval[0] >= .5),
            'by_color': by_color,
            'interpretation': 'Development evidence only. UNKNOWN is excluded from complete-game rates and separately bounded; draws are not wins.'}

def go2_progress(project=None, run_id=None):
    project = go2_project(project)
    active = project / 'reports/ahr_v2/active_run.json'
    if run_id is None and active.is_file():
        value = json.loads(active.read_text()); run_id = value.get('run_id')
    if run_id is None:
        return {'status': 'NO_ACTIVE_TRAINING_RUN', 'message': 'No active AHR v2 run has been recorded.', 'history': []}
    candidates = [project/'reports/ahr_v2'/run_id/'progress.json', project/'checkpoints'/run_id/'progress.json']
    path = next((item for item in candidates if item.is_file()), None)
    if path is None:
        return {'status': 'AWAITING_PROGRESS', 'run_id': run_id, 'message': 'The run has not published a progress record.', 'history': []}
    result = json.loads(path.read_text())
    if result.get('run_id') != run_id: raise RuntimeError('Progress record belongs to another run.')
    result['progress_path'] = str(path.relative_to(project))
    control_path = project/'reports/ahr_v2'/run_id/'control.json'
    if control_path.is_file():
        control = json.loads(control_path.read_text())
        result['active_review'] = control.get('active_review')
        result['development_reviews'] = control.get('reviews', [])
    log_path = project/'reports/ahr_v2'/run_id/'metrics.jsonl'
    by_step = {}
    if log_path.is_file():
        with log_path.open() as stream:
            for line in stream:
                if not line.endswith('\n'): continue  # A concurrently written final line is not yet committed.
                record = json.loads(line)
                data = record.get('data', {})
                step = data.get('optimizer_step')
                if record.get('kind') == 'optimizer' and isinstance(step, int) and step <= result.get('optimizer_step', 0):
                    by_step[step] = data
        result['history'] = [by_step[step] for step in sorted(by_step)[-256:]]
        result['history_source'] = str(log_path.relative_to(project))
    # Records are atomic JSON snapshots. This viewer never restores training RNG or takes its writer lock.
    return result

def go2_progress_display(progress):
    from IPython.display import HTML, display
    import html
    keys = [('status', 'Status'), ('run_id', 'Run'), ('stage', 'Stage'), ('optimizer_step', 'Optimizer updates'),
            ('completed_games', 'Complete self-play games'), ('unknown_games', 'UNKNOWN self-play games'),
            ('replay_train_games', 'Training replay games'), ('replay_train_rows', 'Training positions'),
            ('replay_dev_games', 'Held-out replay games'), ('current_ply', 'Active game ply'),
            ('last_loss', 'Latest joint loss'), ('elapsed_seconds', 'Elapsed seconds'), ('remaining_seconds', 'Budget remaining seconds')]
    rows = []
    for key, label in keys:
        if key not in progress: continue
        value = progress[key]
        if isinstance(value, float): value = f'{value:,.3f}' if key == 'last_loss' else f'{value:,.0f}'
        rows.append('<tr><th style="text-align:left;padding:6px 18px 6px 0">'+html.escape(label)+'</th><td>'+html.escape(str(value))+'</td></tr>')
    display(HTML('<table>'+''.join(rows)+'</table>'))
    active_review = progress.get('active_review')
    if active_review:
        display(HTML('<p><strong>Development review in progress:</strong> '+html.escape(str(active_review.get('eval_id', '')))+
                     '. Training is paused while its pinned checkpoint plays GNU Go.</p>'))
    history = progress.get('history', [])
    points = [row for row in history if isinstance(row, dict) and isinstance(row.get('optimizer_step'), (int, float)) and isinstance(row.get('loss'), (int, float))]
    if points:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.plot([row['optimizer_step'] for row in points], [row['loss'] for row in points], color='#254b87', linewidth=1.5)
        ax.set(title='Recent AHR v2 training loss (up to 256 logged updates)', xlabel='Optimizer updates', ylabel='Joint training loss')
        ax.grid(alpha=.2); fig.tight_layout(); display(fig); plt.close(fig)
    elif progress.get('status') in ('NO_ACTIVE_TRAINING_RUN', 'AWAITING_PROGRESS'):
        display(HTML('<p>No training measurements are available yet.</p>'))
    else:
        display(HTML('<p>This snapshot does not include a loss history; the table shows the published measurements.</p>'))


## Resumable arena engine

An arena pins its model on the first call. Resume with exactly the same `EVAL_ID`; training may advance or prune its original checkpoint because the arena stores its own model snapshot. GNU Go is restored from its seed by replaying every previous `genmove` and checking that the moves agree. Checkpoints are written at search batches and every move/game boundary. Simulation budgets support deterministic recovery; an optional wall-clock search limit preserves the consumed budget but scheduling can change subsequent search work.

In [ ]:
def _go2_eval_player_from_state(state, device):
    with torch.random.fork_rng(devices=[]):
        model = Go2Network(Go2NetConfig(**state['config']['network']))
    model.load_state_dict(state['model']); model.to(device); model.eval()
    for parameter in model.parameters(): parameter.requires_grad_(False)
    if go2_model_identity(model) != state['identity']['model_identity']:
        raise RuntimeError('Arena model snapshot differs from its pinned identity.')
    return model

def _go2_eval_report(state, eval_id, project, status):
    report = {'schema': 'ahr-v2-development-arena-v1', 'eval_id': eval_id,
              'training_run_id': state['identity']['training_run_id'], 'status': status,
              'purpose': state['config']['evaluation']['purpose'], 'model': state['identity']['model_identity'],
              'checkpoint': state['identity']['checkpoint'], 'opponent': state['identity']['opponent'],
              'sources': state['identity']['evaluation_sources'], 'training_sources': state['identity']['training_sources'],
              'config': state['config']['evaluation'], 'summary': go2_eval_summary(state['replay']),
              'games': copy.deepcopy(state['replay']), 'active_game_index': state['sampler']['game_index'],
              'active_ply': len(state['self_play']['trace']) if state['self_play'] else None,
              'updated_at_utc': go2_now(), 'resume': "go2_evaluate(PROJECT, eval_id=" + repr(eval_id) + ")"}
    go2_atomic_json(Path(project)/'reports/ahr_v2/evaluations'/eval_id/'report.json', report)
    return report

def go2_evaluate(project=None, run_id=None, eval_id=None, config=None, which='latest', device='cpu',
                 *, _player=None, _opponent_factory=None, _search_evaluator=None, _max_events=None):
    """Development only. Private injected fixtures are never a playing-strength result."""
    project = go2_project(project)
    if config is not None and isinstance(config, dict): config = Go2EvalConfig(**config)
    if eval_id is None:
        eval_id = 'ahr-v2-dev-' + dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S') + '-' + uuid.uuid4().hex[:6]
    if Path(eval_id).name != eval_id or not eval_id.startswith('ahr-v2-'):
        raise ValueError('Use a local arena run ID beginning ahr-v2-.')
    directory = project/'checkpoints'/eval_id
    with go2_checkpoint_lock(directory):
        state, _ = go2_load_checkpoint(directory)
        if state is None:
            config = config or Go2EvalConfig()
            if (_opponent_factory is not None or _search_evaluator is not None or _player is not None) and config.purpose != 'validation_fixture':
                raise ValueError('Injected fixtures must be labeled validation_fixture.')
            if _player is None:
                if 'go2_load_player' not in globals():
                    raise RuntimeError('Run 04 exports first; no trained-player loader is available.')
                model, metadata = go2_load_player(project, run_id, which=which, device=device)
            else: model, metadata = _player
            run_id = metadata['run_id']
            if config.purpose == 'development':
                if metadata.get('is_smoke') or metadata.get('optimizer_step', 0) <= 0:
                    raise ValueError('Smoke/untrained checkpoints cannot be presented as development strength results.')
                saved_config = metadata.get('config', {})
                if saved_config.get('board_size', config.board_size) != config.board_size or saved_config.get('komi', config.komi) != config.komi:
                    raise ValueError('Arena board/komi differs from the player training configuration.')
            model.eval()
            for parameter in model.parameters(): parameter.requires_grad_(False)
            model_id = go2_model_identity(model)
            if metadata.get('model_identity', model_id) != model_id: raise RuntimeError('Loaded player identity differs.')
            opponent = go2_opponent_pin(config) if _opponent_factory is None else {'name': 'disposable_pass_fixture', 'level': None, 'sha256': go2_digest('pass_fixture_v1')}
            identity = {'schema': 'ahr-v2-pinned-arena-v1', 'training_run_id': run_id,
                        'evaluation_sources': go2_eval_sources(project), 'training_sources': metadata.get('sources', {}),
                        'model_identity': model_id, 'checkpoint': {'file': metadata['checkpoint_file'],
                        'sha256': metadata['checkpoint_sha256'], 'optimizer_step': metadata.get('optimizer_step', 0)},
                        'evaluation_config': dataclasses.asdict(config), 'opponent': opponent}
            state = {'schema': GO2_SCHEMA, 'identity': identity,
                     'model': {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
                     'optimizer': {}, 'scheduler': {}, 'scaler': {}, 'rng': go2_capture_rng(),
                     'sampler': {'game_index': 0}, 'global_step': 0,
                     'config': {'evaluation': dataclasses.asdict(config), 'network': dataclasses.asdict(model.config)},
                     'metrics': {'completed_games': 0}, 'replay': [], 'self_play': None, 'search': None}
            go2_save_checkpoint(directory, state, best=True, resume_cell='05: go2_evaluate(PROJECT, eval_id='+repr(eval_id)+')')
        else:
            saved_config = Go2EvalConfig(**state['config']['evaluation'])
            if config is not None and config != saved_config: raise RuntimeError('Arena configuration changed; resume refused.')
            config = saved_config
            if run_id is not None and run_id != state['identity']['training_run_id']:
                raise RuntimeError('Arena training run changed; resume refused.')
            if go2_eval_sources(project) != state['identity']['evaluation_sources']:
                raise RuntimeError('Arena exported sources changed; resume refused.')
            opponent = state['identity']['opponent']
            if opponent['name'] == 'GNU Go':
                if _opponent_factory is not None or go2_opponent_pin(config) != opponent:
                    raise RuntimeError('GNU Go identity changed; resume refused.')
            elif _opponent_factory is None:
                raise RuntimeError('Validation fixture resume requires its explicit opponent factory.')
            model = _go2_eval_player_from_state(state, device)
        factory = _opponent_factory or (lambda cfg, seed: Go2GTP(cfg, seed, state['identity']['opponent']['executable']))
        event_start = state['global_step']
        def commit(tree=None, game=None, elapsed=None):
            if tree is not None: state['search'] = go2_search_state_dict(tree)
            if game is not None: state['self_play']['state'] = go2_state_to_dict(game)
            if elapsed is not None: state['self_play']['elapsed_seconds'] = elapsed
            state['global_step'] += 1; state['rng'] = go2_capture_rng()
            state['metrics']['completed_games'] = len(state['replay'])
            go2_save_checkpoint(directory, state, resume_cell='05: go2_evaluate(PROJECT, eval_id='+repr(eval_id)+')')
            _go2_eval_report(state, eval_id, project, 'RUNNING')
            if _max_events is not None and state['global_step'] - event_start >= _max_events:
                raise _Go2EvalPause()
        try:
            while state['sampler']['game_index'] < config.games:
                index = state['sampler']['game_index']; color = 1 if index % 2 == 0 else -1
                if state['self_play'] is None:
                    game = go2_initial_state(config.board_size, config.komi)
                    state['self_play'] = {'state': go2_state_to_dict(game), 'trace': [],
                        'seed': config.seed + index//2, 'ahr_color': color, 'elapsed_seconds': 0.0}
                    commit()
                else: game = go2_state_from_dict(state['self_play']['state']); go2_verify_replay(game)
                active = state['self_play']; timer = time.monotonic(); elapsed_base = active['elapsed_seconds']
                unknown_reason = None
                opponent = factory(config, active['seed'])
                try:
                    opponent.replay(active['trace'])
                    while not game.terminal:
                        elapsed = elapsed_base + time.monotonic() - timer
                        if len(active['trace']) >= config.max_plies:
                            unknown_reason = 'PLY_CUTOFF'; break
                        if config.game_time_seconds is not None and elapsed >= config.game_time_seconds:
                            unknown_reason = 'GAME_TIMEOUT'; break
                        if game.to_play == color:
                            search_config = Go2SearchConfig(simulations=config.simulations, batch_size=config.batch_size,
                                seed=config.seed + index*10000 + len(active['trace']), wall_time_seconds=config.move_time_seconds)
                            context = {'purpose': config.purpose, 'eval_id': eval_id, 'game': index, 'ply': len(active['trace'])}
                            tree = None if state['search'] is None else go2_search_from_state_dict(
                                state['search'], model, game, search_config, context=context, project=project)
                            output = go2_search(model, game, search_config, tree=tree, context=context, project=project,
                                evaluator=_search_evaluator,
                                on_batch=lambda tree: commit(tree=tree, game=game, elapsed=elapsed_base+time.monotonic()-timer))
                            action = output['action']
                            opponent.play(game.to_play, action)
                            trace_item = {'kind': 'play', 'color': game.to_play, 'action': action,
                                          'search_simulations': output['completed_simulations'], 'search_status': output['search_status']}
                        else:
                            action = opponent.genmove(game.to_play)
                            trace_item = {'kind': 'genmove', 'color': game.to_play, 'action': action}
                        successor = go2_step(game, action)
                        active['trace'].append(trace_item); state['search'] = None; game = successor
                        commit(game=game, elapsed=elapsed_base+time.monotonic()-timer)
                except _Go2EvalPause: raise
                except (TimeoutError, Go2IllegalMove) as exc:
                    unknown_reason = type(exc).__name__ + ': ' + str(exc)[:200]
                except Go2OpponentError as exc:
                    # Deterministic replay mismatch is an invariant violation, not a game result.
                    if 'regenerated a different' in str(exc): raise
                    unknown_reason = 'OPPONENT_FAILURE: ' + str(exc)[:200]
                finally: opponent.close()
                if game.terminal:
                    result = go2_outcome(game); winner = result['winner']
                    label = 'DRAW' if winner == 0 else 'WIN' if winner == color else 'LOSS'
                    score = color * result['score_black']
                else: label = 'UNKNOWN'; score = None
                finished = {'game_index': index, 'ahr_color': color, 'opponent_seed': active['seed'], 'outcome': label,
                            'unknown_reason': unknown_reason, 'plies': len(active['trace']), 'score_for_ahr': score,
                            'state': go2_state_to_dict(game), 'trace': copy.deepcopy(active['trace'])}
                state['replay'].append(finished); state['sampler']['game_index'] += 1
                state['self_play'] = None; state['search'] = None; commit()
        except _Go2EvalPause:
            return _go2_eval_report(state, eval_id, project, 'PAUSED')
        return _go2_eval_report(state, eval_id, project, 'COMPLETE')

go2_evaluate_player = go2_evaluate


## Short correctness checks

These are disposable fixtures, including a real four-ply GTP smoke check. They are labeled validation data and never reported as player strength or added to training replay. The recovery fixture pauses inside an AHR search and resumes in a fresh kernel.

In [ ]:
class _Go2PassOpponent:
    def __init__(self, config, seed):
        self.config = config; self.state = go2_initial_state(config.board_size, config.komi)
    def play(self, color, action):
        if self.state.to_play != color: raise AssertionError('Fixture opponent turn differs.')
        self.state = go2_step(self.state, action)
    def genmove(self, color):
        action = self.config.board_size*self.config.board_size; self.play(color, action); return action
    def replay(self, trace):
        for item in trace:
            if item['kind'] == 'play': self.play(item['color'], item['action'])
            elif self.genmove(item['color']) != item['action']: raise AssertionError('Fixture replay differs.')
    def close(self): pass

def _go2_pass_evaluator(model):
    def evaluate(states):
        outputs = go2_eval_batch(model, states)
        for state, output in zip(states, outputs):
            output['prior'] = [0.0]*(state.size*state.size) + [1.0]
        return outputs
    return evaluate

def _go2_eval_fixture_player(project):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(50112); model = Go2Network(Go2NetConfig(width=8, blocks=1))
    model.eval()
    metadata = {'run_id': 'ahr-v2-disposable-eval-fixture', 'checkpoint_file': 'fixture-not-a-training-checkpoint',
                'checkpoint_sha256': go2_tensor_digest(model.state_dict()), 'optimizer_step': 0, 'is_smoke': True,
                'model_identity': go2_model_identity(model), 'sources': go2_source_identity(project),
                'config': {'board_size': 3, 'komi': 7.5}}
    return model, metadata

def go2_eval_validation(project=None):
    project = go2_project(project); torch.set_num_threads(2)
    checks = {}
    for size in (3, 9, 19):
        for action in range(size*size+1): assert go2_gtp_action(go2_gtp_vertex(action, size), size) == action
    checks['gtp_coordinates_including_skipped_I_and_pass'] = True
    for vertex in ('I9', 'Z9', 'A0', 'A20', 'RESIGN'):
        try: go2_gtp_action(vertex, 9)
        except (ValueError, RuntimeError): pass
        else: raise AssertionError('Invalid or resigned GTP action accepted.')
    checks['invalid_coordinates_and_resignation_rejected'] = True
    inputs = [{'outcome': label, 'ahr_color': 1 if i % 2 == 0 else -1} for i, label in enumerate(('WIN', 'LOSS', 'DRAW', 'UNKNOWN'))]
    summary = go2_eval_summary(inputs)
    assert summary['attempts'] == 4 and summary['complete_games'] == 3 and summary['unknown'] == 1
    assert summary['win_rate_complete_games'] == 1/3 and summary['win_rate_all_attempts_lower_bound'] == .25
    assert summary['win_rate_all_attempts_upper_bound'] == .5 and not summary['observed_half_target']
    assert abs(go2_wilson(1,3)[0] - .06149194472039621) < 1e-12
    assert abs(go2_wilson(1,3)[1] - .7923403991979522) < 1e-12
    assert go2_wilson(0,0) == [None, None]
    checks['UNKNOWN_denominators_draws_and_Wilson'] = True
    token = uuid.uuid4().hex[:10]; full_id = 'ahr-v2-eval-fixture-'+token+'-full'; resume_id = 'ahr-v2-eval-fixture-'+token+'-resume'
    cfg = Go2EvalConfig(games=2, board_size=3, simulations=4, batch_size=2, max_plies=8,
                       game_time_seconds=None, purpose='validation_fixture')
    player = _go2_eval_fixture_player(project)
    arguments = {'config': cfg, '_player': player, '_opponent_factory': _Go2PassOpponent,
                 '_search_evaluator': _go2_pass_evaluator(player[0])}
    full = go2_evaluate(project, eval_id=full_id, **arguments)
    partial = go2_evaluate(project, eval_id=resume_id, _max_events=2, **arguments)
    assert partial['status'] == 'PAUSED'
    saved, _ = go2_load_checkpoint(project/'checkpoints'/resume_id)
    assert saved['search'] is not None and saved['search']['completed_simulations'] == 2
    checks['active_search_and_game_checkpointed'] = True
    fixture = nbformat.v4.new_notebook(metadata={'kernelspec': {'name': 'ahr-go', 'language': 'python', 'display_name': 'AHR Go'}})
    setup = "from pathlib import Path\nimport nbformat\nPROJECT=Path(" + repr(str(project)) + ")\nfor name in ('00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','05_evaluation_and_progress.ipynb'):\n    for cell in nbformat.read(PROJECT/'experiments/ahr_v2'/name,4).cells:\n        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())\ntorch.set_num_threads(2)\n"
    fixture.cells = [nbformat.v4.new_code_cell(setup), nbformat.v4.new_code_cell(
        "player=_go2_eval_fixture_player(PROJECT)\nreport=go2_evaluate(PROJECT,eval_id=" + repr(resume_id) + ",_opponent_factory=_Go2PassOpponent,_search_evaluator=_go2_pass_evaluator(player[0]))\nassert report['status']=='COMPLETE'\n")]
    fixture_path = project/'reports/ahr_v2/fixtures'/('eval-resume-'+token+'.ipynb')
    fixture_path.parent.mkdir(parents=True, exist_ok=True); nbformat.write(fixture, fixture_path)
    subprocess.run([sys.executable, '-m', 'jupyter', 'nbconvert', '--execute', '--to', 'notebook', '--inplace',
                    '--ExecutePreprocessor.kernel_name=ahr-go', '--ExecutePreprocessor.timeout=120', str(fixture_path)],
                   capture_output=True, text=True, check=True)
    resumed = json.loads((project/'reports/ahr_v2/evaluations'/resume_id/'report.json').read_text())
    assert full['games'] == resumed['games'] and full['summary'] == resumed['summary']
    assert full['summary']['wins'] == 1 and full['summary']['losses'] == 1 and full['summary']['unknown'] == 0
    checks['fresh_kernel_exact_games_and_outcomes'] = True
    latest, _ = go2_load_checkpoint(project/'checkpoints'/resume_id)
    assert len(json.loads((project/'checkpoints'/resume_id/'manifest.json').read_text())['entries']) == 3
    checks['latest_two_plus_best_retained'] = True
    with Go2GTP(Go2EvalConfig(games=2, max_plies=4), 71012) as engine:
        opening = engine.genmove(1); engine.play(-1, 81); followup = engine.genmove(1)
        trace = [{'kind':'genmove','color':1,'action':opening}, {'kind':'play','color':-1,'action':81},
                 {'kind':'genmove','color':1,'action':followup}]
    with Go2GTP(Go2EvalConfig(games=2, max_plies=4), 71012) as engine: engine.replay(trace)
    state = go2_initial_state();
    for item in trace: state = go2_step(state, item['action'])
    go2_verify_replay(state)
    checks['real_GNU_Go_seed_and_genmove_replay'] = True
    smoke_cfg = Go2EvalConfig(games=2, simulations=1, batch_size=1, max_plies=4,
                              game_time_seconds=30, purpose='validation_fixture')
    smoke = go2_evaluate(project, eval_id='ahr-v2-eval-gnu-smoke-'+token, config=smoke_cfg, _player=player)
    assert smoke['summary']['unknown'] == 2 and smoke['summary']['complete_games'] == 0
    assert all(item['unknown_reason'] == 'PLY_CUTOFF' for item in smoke['games'])
    checks['real_neural_GNU_Go_smoke_cutoffs_remain_UNKNOWN'] = True
    # Same manifest with a corrupt payload must not silently restart the arena.
    manifest = json.loads((project/'checkpoints'/resume_id/'manifest.json').read_text())
    payload = project/'checkpoints'/resume_id/manifest['latest']; original = payload.read_bytes()
    try:
        payload.write_bytes(original+b'corrupt')
        try: go2_load_checkpoint(project/'checkpoints'/resume_id)
        except RuntimeError: pass
        else: raise AssertionError('Arena corrupt checkpoint accepted.')
    finally: payload.write_bytes(original)
    checks['corrupt_arena_checkpoint_rejected'] = True
    receipt = {'status':'PASS', 'checks':checks, 'sources':go2_eval_sources(project),
               'opponent':go2_opponent_pin(smoke_cfg), 'full_fixture':full_id, 'resume_fixture':resume_id,
               'fresh_kernel_notebook':str(fixture_path.relative_to(project)), 'gnu_smoke':smoke['eval_id'],
               'scope':'Correctness fixtures only. No trained-player strength claim or training data.',
               'validated_at_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/evaluation_validation.json', receipt)
    return receipt


In [ ]:
validation = go2_eval_validation(PROJECT)
from IPython.display import HTML, display
display(HTML('<table><tr><th>Validation check</th><th>Result</th></tr>'+''.join('<tr><td>'+name.replace('_',' ')+'</td><td>PASS</td></tr>' for name, passed in validation['checks'].items())+'</table>'))

## Current training progress

The loss chart reads committed optimizer records through the published checkpoint step and keeps the latest 256 distinct updates. Training loss measures learning on replay; it is not a Go win rate. During an independent development review, the table explicitly shows that training is paused.

In [ ]:
progress = go2_progress(PROJECT, RUN_ID)
go2_progress_display(progress)

## Run or resume a development review

Set `RUN_EVALUATION=True` above and choose the same `EVAL_ID` to resume. A new ID pins a new checkpoint. The notebook refuses to report an untrained or smoke checkpoint as a development strength result. An interrupted evaluation preserves its own frozen player; it does not reload newer training weights.

In [ ]:
EVAL_CONFIG = Go2EvalConfig(games=8, simulations=32, max_plies=400, game_time_seconds=180, move_time_seconds=2)
if RUN_EVALUATION:
    arena = go2_evaluate(PROJECT, RUN_ID, EVAL_ID, EVAL_CONFIG, device=DEVICE)
    EVAL_ID = arena['eval_id']
    display(arena['summary'])
else:
    print('Arena launch is off. Short correctness fixtures passed; no playing-strength measurement has been made.')

## Exact resume and next notebook

For a paused review, execute setup and export definition cells, then `go2_evaluate(PROJECT, eval_id="your-existing-eval-id")`. The same notebook sources and GNU Go executable must remain available. A corrupt checkpoint, changed sources, changed configuration or changed opponent is an error; the notebook never silently starts over. One writer owns each arena ID.

For training progress, rerun only the progress cell. A final evaluation must use a separately reserved match schedule; this notebook is development only. Open **06 — Play against AHR** to interact with the active trained checkpoint. No long arena is automatically launched by Run All.